# ENES User Story: climate data analysis and provenance management

In this notebook we compute the **seasonal climatology** and the **sesonal anomalies** for a particular season of interest.

We exploit the **prov** library (https://github.com/trungdong/prov) to create the corresponding provenance document, which is then stored to the **ENES provenance service**.

Authentication and authentication are managed through the **EOSC AAI**.

In [ ]:
import intake
import xarray as xr
import panel as pn
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
from cartopy.mpl.geoaxes import GeoAxes
from cartopy.util import add_cyclic_point
import requests, json
import warnings
import requests
from pystac_client import Client
warnings.filterwarnings("ignore")

In [ ]:
%pip install networkx

## Accessing CMIP6 Data

Datasets are stored in ZARR format leveraging the object storage service provided by CESNET: https://object-store.brno.openstack.cloud.e-infra.cz/

We use the **pySTAC** library to execute queries against the ENES STAC (SpatioTemporal Asset Catalogs) catalog and easily search, discover and access data. 

In [ ]:
url_stac = "https://api.eneslab.pilot.eosc-beyond.eu"

variable = "tas"
frequency = "Amon"
model = "CMCC-ESM2"

Let's pick data about a specific model (`CMCC-ESM2`) and climate variable (`tas`) simulated at `daily` frequency for `future scenario` experiments.

In [ ]:
catalog = Client.open(url_stac)

search = catalog.search(
    query={
        "cmip6:source_id": {"eq": model},
        "cmip6:variable_id": {"eq": variable},
        "cmip6:table_id": {"eq": frequency},
        "cmip6:experiment_id": {"ne": "historical"},
    },
    method="GET"
)
items = list(search.items())
items

Let's explore the metadata for one of the items

In [ ]:
print(json.dumps(items[0].to_dict(), indent=2, ensure_ascii=False))

## Compute a Seasonal Climatology and the Seasonal Anomaly

### Seasonal Climatology

For each experiment, let's compute the seasonal climatology, which refers to the analysis of long-term average atmospheric and surface conditions (such as temperature, precipitation, and wind) computed separately for each of the four meteorological seasons (e.g., DJF, MAM, JJA, SON).
By averaging data over many years, it highlights the characteristic “typical” climate for each season and helps identify deviations or trends relative to those seasonal norms.

### Seasonal Anomaly

Then, to analyze the climate variability, it is common to compare current observations with long-term averages. A **seasonal anomaly** measures the difference between observed seasonal conditions and the corresponding climatological mean.

In this case, we compute the temperature anomaly for the **summer season (JJA) of a given year** using the following steps:

1. **Extract data for the summer period** (June–August) of the target year from the input dataset.
2. **Aggregate the seasonal data** by computing the mean temperature across the selected months.
3. **Select the climatological mean** for the summer season (JJA) from a precomputed seasonal climatology.
4. **Subtract the climatology from the seasonal mean**, yielding the anomaly field:

$$
\text{Anomaly}_{\text{year},\text{JJA}} = \overline{T}_{\text{year},\text{JJA}} - \overline{T}_{\text{Climatology},\text{JJA}}
$$

This anomaly highlights regions where temperatures during the summer were **above or below normal**, providing insight into unusual climate behavior or extreme events.

As an example, we will consider the Europe domain.

### Provenance generation

During the computation, we will keep track of all the performed steps, thus generating a structured and well-formed provenance document compliant with the **W3C PROV standard** (https://www.w3.org/TR/prov-overview/).

In [ ]:
north = 71.59
south = 32.13
east = 48.04
west = -30.08

from prov.model import ProvDocument
d1 = ProvDocument()
d1.add_namespace('yprov', 'https://github.com/HPCI-Lab/yProv/')

def compute_seasonal_climatology(CMIP6):
    seasonal_climatology = CMIP6.tas.groupby("time.season").mean() - 273.15
    return seasonal_climatology.compute()

def compute_seasonal_anomalies(seasonal_climatology, CMIP6):   
    season_of_interest = CMIP6.sel(time=slice("2025-06-01", "2025-08-31")) # year:2025; season:JJA
    season_of_interest_mean = season_of_interest.tas.mean(dim='time') - 273.15
    soi_climatology = seasonal_climatology.sel(season='JJA')
    anomaly = season_of_interest_mean - soi_climatology
    return anomaly

def save_plot(result, title):
    fig = plt.figure(figsize=(10, 5), dpi=100)

    projection = ccrs.PlateCarree()
    ax = plt.axes(projection=projection)

    ax.coastlines()

    gl = ax.gridlines(crs=projection, draw_labels=True, linewidth=1, color='black', alpha=0.9, linestyle=':')
    gl.xlabels_top = False
    gl.ylabels_right = False

    var = result.values
    lat = result.lat
    lon = result.lon
    var = np.reshape(var, (len(lat), len(lon)))

    var_cyclic, lon_cyclic = add_cyclic_point(var, coord=np.asarray(lon))
    x, y = np.meshgrid(lon_cyclic,lat)

    levStep = (np.nanmax(var)-np.nanmin(var))/20
    clevs = np.arange(np.nanmin(var),np.nanmax(var)+levStep,levStep)

    cnplot = ax.contourf(x, y, var_cyclic, clevs,transform=projection, cmap='coolwarm')  # transform keyword provided
    plt.colorbar(cnplot,ax=ax)

    ax.set_aspect('auto', adjustable=None)

    plt.title(title)
    
    plt.savefig(title+".png", bbox_inches='tight')
    plt.close(fig)

In [ ]:
experiments = []
for item in items:
    path = item.assets["zarr"].href
    experiment = item.properties['experiment_id']
    experiments.append(experiment)
    print("Experiment: "+experiment)
    
    CMIP6 = xr.open_zarr(path, consolidated=True)
    CMIP6 = CMIP6.sel(lat=slice(south, north), lon=slice(west, east))

    input_subset = d1.entity('yprov:'+path.split('vo.enes.org-cmip6/')[-1])
    subset_activity = d1.activity('yprov:subset')
    output_subset = d1.entity('yprov:'+path.split('vo.enes.org-cmip6/')[-1]+'_subset')
    d1.wasGeneratedBy(output_subset,subset_activity)
    d1.wasDerivedFrom(output_subset,input_subset)
    d1.used(subset_activity,input_subset)

    seasonal_climatology = compute_seasonal_climatology(CMIP6)

    season = pn.widgets.Select(name='Season', options=list(seasonal_climatology.season.values))

    seasonal_climatology_activity = d1.activity('yprov:seasonal_climatology')
    output_seasonal_climatology_activity = d1.entity('yprov:'+path.split('vo.enes.org-cmip6/')[-1]+'_seasonal_climatology')
    d1.wasGeneratedBy(output_seasonal_climatology_activity,seasonal_climatology_activity)
    d1.wasDerivedFrom(output_seasonal_climatology_activity,output_subset)
    d1.used(seasonal_climatology_activity,output_subset)

    anomaly = compute_seasonal_anomalies(seasonal_climatology, CMIP6)

    anomaly_activity = d1.activity('yprov:anomaly')
    output_anomaly_activity = d1.entity('yprov:'+path.split('vo.enes.org-cmip6/')[-1]+'_anomaly')

    d1.used(anomaly_activity,output_seasonal_climatology_activity)
    d1.used(anomaly_activity,output_subset)
    d1.wasGeneratedBy(output_anomaly_activity,anomaly_activity)

    d1.wasDerivedFrom(output_anomaly_activity,output_subset)
    d1.wasDerivedFrom(output_anomaly_activity,output_seasonal_climatology_activity)

    export_activity = d1.activity('yprov:export')
    output_export = d1.entity('yprov:'+'anomaly_2025_JJA_{0}.zarr'.format(experiment))
    d1.used(export_activity,output_anomaly_activity)
    d1.wasGeneratedBy(output_export,export_activity)
    d1.wasDerivedFrom(output_export,output_anomaly_activity)
    
    save_plot(anomaly,'anomaly_2025_JJA_{0}'.format(experiment))

## Data visualization

In [ ]:
from PIL import Image as PILImage
frames = []
for e in experiments:
    frames.append(PILImage.open("anomaly_2025_JJA_"+e+".png"))
    
frames[0].save(
    "anomaly_2025_JJA.gif",
    save_all=True,
    append_images=frames[1:],
    duration=1000,
    loop=0
)
from IPython.display import Image
Image('anomaly_2025_JJA.gif')

### Generate a png image from the provenance document

In [ ]:
from prov.dot import prov_to_dot
dot = prov_to_dot(d1)
dot.write_png('provenance_anomaly.png')

In [ ]:
from IPython.display import Image
Image('provenance_anomaly.png')

### Export the provenance document to a JSON file

In [ ]:
d1.serialize('provenance_anomaly.json')

### Upload the provenance document to the ENES Provenance service

This requires obtaining a token from https://aai-demo.egi.eu/token to be used in the PUT request.

Then, the provenance service will make use of the **Token Introspection Process** for managing authentication and authorization. Specifically:
1. The API extracts the Bearer token from the Authorization header
2. It sends the token to the introspection endpoint (https://enes-proxy.pilot.eosc-beyond.eu/auth/realms/enes/protocol/openid-connect/token/introspect)
3. It checks the introspection response to ensure that the token is valid and includes the required entitlements.

In [ ]:
token = "..."

In [ ]:
yprov_endpoint = "http://yprov.disi.unitn.it:8004"

In [ ]:
from pathlib import Path
import json
import requests

base_url = yprov_endpoint.rstrip("/")
file_path = Path("./provenance_anomaly.json")

# Validate the source file before uploading
original = file_path.read_bytes()
if not original:
    raise ValueError("The file is empty.")
expected_document = json.loads(original)

headers = {"Authorization": f"Bearer {token}"}

metadata = {
    "title": "ENES User Story - Seasonal Anomaly using CMIP6 data",
    "author": "Fabrizio Antonio",
}
    
with file_path.open("rb") as f:
    response = requests.post(
        f"{base_url}/documents",
        headers=headers,
        files={
            "document_file": (
                file_path.name,
                f,
                "application/json",
            )
        },
        params={
            "document_metadata": json.dumps(metadata)
        },
        timeout=120,
    )
    
print("Upload:", response.status_code, response.text)
response.raise_for_status()
record = response.json()

### Attach artifact

In [ ]:
import os, requests
file_path = 'anomaly_2025_JJA.gif'
filename = os.path.basename(file_path)
headers = {"Authorization": f"Bearer {token}"}

r = requests.post(f"{yprov_endpoint}/artifacts/upload/url", params={"filename": filename}, headers=headers)
r.raise_for_status()
upload_info = r.json()
upload_url = upload_info["upload_url"]

with open(file_path, "rb") as f:
    files = {"document_file": (filename, f, "application/octet-stream")}
    r2 = requests.put(upload_url, files=files, headers=headers)
        
r2.raise_for_status()
print(r2.json())